# TFR pipeline на реальных данных (`lib.*`)

Ноутбук воспроизводит refactored-пайплайн на реальном `tfr_*.fif`:

- загрузка данных как в `transformer_17_03_26.ipynb` (`read_tfrs` -> `y` из `events[:, 2]` -> crop/normalization)
- запуск Optuna через `lib.optuna.*` с хранением trial'ов в SQLite
- сохранение `y_true` / `y_hat` по каждому CV-фолду для последующего анализа

Базы Optuna сохраняются в папку с текущей датой внутри корневой папки данных.

In [ ]:
import gc
import json
import sys
from datetime import date
from pathlib import Path

import mne
import numpy as np
import optuna
import torch
import yaml
from torch.utils.data import DataLoader

root = Path.cwd().resolve().parent
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from lib.data import TFRDataset
from lib.models.alexnet import AlexNetTFR
from lib.models.tfr_transformer import TFRTransformerWrapper
from lib.optuna import (
    attrs_fn,
    loss_slope,
    make_objective_engine,
    make_splits_fn_factory,
    objectives_fn,
    params_fn_factory,
    params_fn_factory_transformer,
    run_fold_fn_factory,
)
from lib.training.epochs import eval_one_epoch_f1_macro, train_one_epoch
from utils.normalisation import normalize_tfr_robust

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("project root:", root)
print("device:", device)

## 1. Загрузка реальных данных (`tfr_*.fif`) + папка БД с сегодняшней датой

In [ ]:
# Ожидаем структуру:
#   ├─ NeuronDeCo/
#   │   ├─ patients.yaml
#   │   └─ notebooks/
# PreprocessedData/

#   └─ specs_with_car/
#       └─ tfr_s11.fif

# Рабочая директория: .../Pirogov/NeuronDeCo/notebooks
NB_DIR = Path.cwd().resolve()
PROJECT_ROOT = NB_DIR.parent
PIROGOV_ROOT = PROJECT_ROOT.parent
PREPROCESSED_ROOT = PIROGOV_ROOT / "PreprocessedData"

patients_candidates = [
    PREPROCESSED_ROOT / "patients.yaml",  # приоритет: как вы просили
    PROJECT_ROOT / "patients.yaml",       # fallback на старый вариант
]
patients_yaml = next((p for p in patients_candidates if p.exists()), None)
if patients_yaml is None:
    raise FileNotFoundError(
        "patients.yaml not found. Checked: " + ", ".join(str(p) for p in patients_candidates)
    )

with open(patients_yaml, encoding="utf-8") as f:
    test = yaml.safe_load(f)
pat_config = test["default"]

# time_frequency_file = Path(pat_config["time_frequency_file"])
tfr_candidates = [
    PREPROCESSED_ROOT / "specs_with_car" / "tfr_s11.fif",  # приоритет: PreprocessedData
    PROJECT_ROOT / "specs_with_car" / "tfr_s11.fif",       # fallback
]
time_frequency_file = next((p for p in tfr_candidates if p.exists()), None)
if time_frequency_file is None:
    raise FileNotFoundError(
        "TFR file not found. Checked: " + ", ".join(str(p) for p in tfr_candidates)
    )

min_f, max_f, min_t, max_t = (
    pat_config["min_f"],
    pat_config["max_f"],
    pat_config["min_t"],
    pat_config["max_t"],
)

DATA_ROOT = time_frequency_file.parent
DATE_TAG = date.today().isoformat()
OPTUNA_DB_DIR = DATA_ROOT / DATE_TAG
OPTUNA_DB_DIR.mkdir(parents=True, exist_ok=True)

print("notebooks dir:", NB_DIR)
print("Pirogov root:", PIROGOV_ROOT)
print("PreprocessedData root:", PREPROCESSED_ROOT)
print("patients.yaml:", patients_yaml)
print("TFR file:", time_frequency_file)
print("Optuna DB dir:", OPTUNA_DB_DIR)
print("config slices:", (min_f, max_f, min_t, max_t))

tfr = mne.time_frequency.read_tfrs(str(time_frequency_file))[0]
y = np.where(tfr.events[:, 2] == 9, 1, 0).astype(np.int64)

# X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, min_f:max_f, min_t:max_t]
X = normalize_tfr_robust(tfr.crop(tmin=0.0, tmax=1.0).data)[:, :, :-50, :].astype(np.float32)

del tfr
gc.collect()  # Теперь память гарантировано свободна

n, c, f, t = X.shape
num_classes = int(np.unique(y).shape[0])
print("X shape:", X.shape)
print("y shape:", y.shape)
print("num_classes:", num_classes)

## 2. Optuna (AlexNet) c сохранением в SQLite

In [ ]:
seed = 42
cv = True
test_size = 0.2
cv_aggregate = "median"
max_epochs = 100
patience = 6
n_trials = 30

alex_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_alexnet.db"
alex_storage = f"sqlite:///{alex_db}"

objective_alex = make_objective_engine(
    X=X,
    y=y,
    make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=AlexNetTFR,
        device=device,
        max_epochs=max_epochs,
        patience=patience,
        TFRDataset=TFRDataset,
        DataLoader=DataLoader,
        train_one_epoch=train_one_epoch,
        eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope,
    ),
    aggregate_mode=cv_aggregate,
    params_fn=params_fn_factory(in_channels=c, num_classes=num_classes),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)

study_a = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=seed),
    storage=alex_storage,
    study_name=f"{time_frequency_file.stem}_alexnet",
    load_if_exists=True,
)
study_a.optimize(objective_alex, n_trials=n_trials, show_progress_bar=True)

print("AlexNet DB:", alex_db)
study_a.best_trials[0].values, study_a.best_trials[0].params

## 3. Optuna (Transformer) c сохранением в SQLite

In [ ]:
tr_db = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_transformer.db"
tr_storage = f"sqlite:///{tr_db}"

objective_tr = make_objective_engine(
    X=X,
    y=y,
    make_splits_fn=make_splits_fn_factory(test_size=test_size, seed=seed, cv=cv),
    run_fold_fn=run_fold_fn_factory(
        ModelCls=TFRTransformerWrapper,
        device=device,
        max_epochs=max_epochs,
        patience=patience,
        TFRDataset=TFRDataset,
        DataLoader=DataLoader,
        train_one_epoch=train_one_epoch,
        eval_one_epoch_f1_macro=eval_one_epoch_f1_macro,
        loss_metric=loss_slope,
    ),
    aggregate_mode=cv_aggregate,
    params_fn=params_fn_factory_transformer(
        num_classes=num_classes,
        seq_len=t,
    ),
    objectives_fn=objectives_fn,
    attrs_fn=attrs_fn,
)

study_t = optuna.create_study(
    directions=["maximize", "minimize"],
    sampler=optuna.samplers.NSGAIISampler(seed=seed + 1),
    storage=tr_storage,
    study_name=f"{time_frequency_file.stem}_transformer",
    load_if_exists=True,
)
study_t.optimize(objective_tr, n_trials=n_trials, show_progress_bar=True)

print("Transformer DB:", tr_db)
study_t.best_trials[0].values, study_t.best_trials[0].params

## 4. Сохранение `y_true` / `y_hat` по каждому CV-фолду

In [ ]:
def pick_best_trial(study: optuna.Study):
    # Для multi-objective выбираем trial с максимальным f1 и затем минимальным slope
    complete = [t for t in study.trials if t.values is not None]
    if not complete:
        raise RuntimeError("No complete trials in study")
    return sorted(complete, key=lambda tr: (-tr.values[0], tr.values[1]))[0]


@torch.no_grad()
def predict_fold(model, loader, device):
    model.eval()
    y_true, y_hat = [], []
    for xb, yb in loader:
        xb = xb.to(device)
        logits = model(xb)
        pred = logits.argmax(dim=1).cpu().numpy()
        y_hat.append(pred)
        y_true.append(yb.cpu().numpy())
    return np.concatenate(y_true), np.concatenate(y_hat)


def _restore_model_params_for_runtime(model_params: dict):
    params = dict(model_params)

    pooling = params.get("pooling")
    if isinstance(pooling, dict) and pooling.get("type") == "SeqPool":
        params["pooling"] = pooling.get("mode", "mean")

    preprocess = params.get("preprocess")
    if isinstance(preprocess, dict):
        preprocess_type_to_key = {
            "TFRToSeqFlatten": "flatten",
            "TFRToSeqChannelConvCollapse": "channel_conv",
            "TFRToSeqFTPlaneConvCollapse": "ft_plane_conv",
            "TFRToSeqPixelWeightCollapse": "pixel_weight",
        }
        p_type = preprocess.get("type")
        if p_type in preprocess_type_to_key:
            params["preprocess"] = preprocess_type_to_key[p_type]
            if p_type == "TFRToSeqFTPlaneConvCollapse":
                if "kernel_freq" in preprocess:
                    params["kernel_freq"] = int(preprocess["kernel_freq"])
                if "kernel_time" in preprocess:
                    params["kernel_time"] = int(preprocess["kernel_time"])

    return params


def collect_cv_predictions(ModelCls, best_params, *, file_tag: str):
    splits = make_splits_fn_factory(test_size=test_size, seed=seed, cv=True)(X, y)
    out = {
        "file": str(time_frequency_file),
        "model": file_tag,
        "seed": seed,
        "folds": [],
    }

    for sp in splits:
        p_tr_ds = best_params["tr_dataset"]
        p_vl_ds = best_params["vl_dataset"]
        p_tr_ld = best_params["tr_loader"]
        p_vl_ld = best_params["vl_loader"]

        train_ds = TFRDataset(sp.X_train, sp.y_train, **p_tr_ds)
        val_ds = TFRDataset(sp.X_val, sp.y_val, **p_vl_ds)
        train_loader = DataLoader(train_ds, **p_tr_ld)
        val_loader = DataLoader(val_ds, **p_vl_ld)

        model_kwargs = _restore_model_params_for_runtime(best_params["model"])
        model = ModelCls(**model_kwargs).to(device)
        optimizer = torch.optim.AdamW(model.parameters(), **best_params["optimizer"])

        best_f1 = -1.0
        bad = 0
        for _epoch in range(max_epochs):
            _ = train_one_epoch(model, train_loader, optimizer, device)
            _, va_f1 = eval_one_epoch_f1_macro(model, val_loader, device)
            if va_f1 > best_f1:
                best_f1 = float(va_f1)
                bad = 0
            else:
                bad += 1
                if bad >= patience:
                    break

        y_true_fold, y_hat_fold = predict_fold(model, val_loader, device)
        out["folds"].append(
            {
                "split": sp.name,
                "best_f1": best_f1,
                "y_true": y_true_fold.tolist(),
                "y_hat": y_hat_fold.tolist(),
            }
        )

    save_path = OPTUNA_DB_DIR / f"{time_frequency_file.stem}_{file_tag}_cv_predictions.json"
    with open(save_path, "w", encoding="utf-8") as f:
        json.dump(out, f, ensure_ascii=False, indent=2)

    return save_path, out


best_a = pick_best_trial(study_a)
best_t = pick_best_trial(study_t)

pred_path_a, pred_data_a = collect_cv_predictions(
    AlexNetTFR,
    best_a.user_attrs["params"],
    file_tag="alexnet",
)
pred_path_t, pred_data_t = collect_cv_predictions(
    TFRTransformerWrapper,
    best_t.user_attrs["params"],
    file_tag="transformer",
)

print("Saved:", pred_path_a)
print("Saved:", pred_path_t)
print("AlexNet folds:", len(pred_data_a["folds"]))
print("Transformer folds:", len(pred_data_t["folds"]))

In [ ]:
from sklearn.metrics import confusion_matrix


def print_confusion_from_json(json_path):
    json_path = Path(json_path)
    with open(json_path, "r", encoding="utf-8") as f:
        payload = json.load(f)

    print(f"\n=== Confusion matrices: {json_path.name} ===")
    all_true, all_hat = [], []

    for fold in payload.get("folds", []):
        y_true = np.asarray(fold["y_true"], dtype=np.int64)
        y_hat = np.asarray(fold["y_hat"], dtype=np.int64)
        labels = sorted(set(y_true.tolist()) | set(y_hat.tolist()))
        cm = confusion_matrix(y_true, y_hat, labels=labels)

        all_true.append(y_true)
        all_hat.append(y_hat)

        print(f"\n[{fold['split']}] labels={labels}")
        print(cm)

    if all_true:
        y_true_all = np.concatenate(all_true)
        y_hat_all = np.concatenate(all_hat)
        labels_all = sorted(set(y_true_all.tolist()) | set(y_hat_all.tolist()))
        cm_all = confusion_matrix(y_true_all, y_hat_all, labels=labels_all)
        print(f"\n[ALL FOLDS] labels={labels_all}")
        print(cm_all)


print_confusion_from_json(pred_path_a)
print_confusion_from_json(pred_path_t)